# Register GGUF To Ollama
Use this notebook to register an existing local GGUF file into the installed Windows Ollama server.


In [2]:
# Set GGUF and Ollama path + settings.
from pathlib import Path

GGUF_PATH = Path(r"C:\Users\Desktop\llm_models\gguf_cache\Turkish-Gemma-9b-T1.Q4_K_M.gguf")
OLLAMA_MODEL_NAME = "turkish-gemma-t1-q4km"
OLLAMA_NUM_CTX = 4096
OLLAMA_HOST = "127.0.0.1:11434"
OLLAMA_BASE = f"http://{OLLAMA_HOST}"

# Ollama serves its local daemon over loopback HTTP on this machine.

print("GGUF_PATH =", GGUF_PATH)
print("OLLAMA_MODEL_NAME =", OLLAMA_MODEL_NAME)


GGUF_PATH = C:\Users\Desktop\llm_models\gguf_cache\Turkish-Gemma-9b-T1.Q4_K_M.gguf
OLLAMA_MODEL_NAME = turkish-gemma-t1-q4km


In [ ]:
# Check Ollama server is reachable.
import urllib.request

with urllib.request.urlopen(f"{OLLAMA_BASE}/api/version", timeout=5) as response:
    print(response.read().decode("utf-8"))


{"version":"0.17.4"}


In [4]:
# Register the GGUF as an Ollama model.
import subprocess
import tempfile

show = subprocess.run(["ollama", "show", OLLAMA_MODEL_NAME], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if show.returncode == 0:
    print(f"Model already registered: {OLLAMA_MODEL_NAME}")
else:
    with tempfile.NamedTemporaryFile("w", suffix=".modelfile", delete=False) as f:
        modelfile_path = Path(f.name)
        f.write(f"FROM {GGUF_PATH}\n")
        f.write(f"PARAMETER num_ctx {OLLAMA_NUM_CTX}\n")

    subprocess.run(["ollama", "create", OLLAMA_MODEL_NAME, "-f", str(modelfile_path)], check=True)
    modelfile_path.unlink()
    print(f"Registered model: {OLLAMA_MODEL_NAME}")

subprocess.run(["ollama", "list"], check=True)


Model already registered: turkish-gemma-t1-q4km


CompletedProcess(args=['ollama', 'list'], returncode=0)

In [10]:
# Run a quick generation on Ollama API.
import requests

payload = {
    "model": OLLAMA_MODEL_NAME,
    "prompt": "Selam!",
    "stream": False,
    "options": {"num_ctx": OLLAMA_NUM_CTX},
}

resp = requests.post(f"{OLLAMA_BASE}/api/generate", json=payload, timeout=600)
resp.raise_for_status()
data = resp.json()
print((data.get("response") or "").strip())


Bugün sizlere **"Yazılım Geliştirme Süreci Modelleri"** konusunu anlatacağım. Bu konu, yazılım mühendisliğinin temel taşlarından biridir ve projelerin nasıl planlandığını, yönetildiğini ve geliştirildiğini gösterir. 

### Neden Yazılım Geliştirme Süreçleri Önemli?
- **Kalite Güvencesi:** Kodun daha güvenilir olmasını sağlar.
- **Verimlilik:** Kaynakların (zaman, para) etkin kullanımı.
- **Risk Yönetimi:** Potansiyel sorunların önceden tespiti.
- **İletişim:** Tüm ekip üyeleri arasında uyum.

### Popüler Süreç Modelleri:
1. **Şelale Modeli (Waterfall)**  
   - **Nasıl Çalışır?** Ardışık aşamalar halinde ilerleme: Gereksinimler → Tasarım → Kodlama → Test → Bakım.  
   - **Artıları:** Basit ve doğrusal. Resmi dokümantasyon ağırlıklı.  
   - **Eksileri:** Değişikliklere esnek değil. Hata erken aşamalarda fark edilirse bile, geri dönüş maliyetli.  
   - **İyi Olduğu Yerler:** Gereksinimlerin net olduğu, sabit projeler (örn: gömülü sistem yazılımları).

2. **Artımlı Geliştirme (Incremental D

In [9]:
# Show the loaded model and GPU/CPU split.
import subprocess

result = subprocess.run(["ollama", "ps"], check=True, capture_output=True, text=True)
print(result.stdout)


NAME                            ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
turkish-gemma-t1-q4km:latest    919e8efccc41    7.7 GB    100% GPU     4096       3 minutes from now    

